### Install & Import required libraries

In [1]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import faiss
import torch

### Define search query & top-k results to retrieve

In [2]:
user_query = "An affordable hotel with a view of the Eiffel Tower"
top_k = 5

### Load Dataset

In [3]:
dataset = load_dataset("traversaal-ai-hackathon/hotel_datasets")

extract the 'train' split and convert it into a pandas DataFrame

In [4]:
df = dataset["train"].to_pandas()

### Filter for Hotels only in Paris and clean the dataset from erroneous values

In [5]:
df_paris = df.loc[
    (df.locality == "Paris") &
    (df.review_text.str.strip() != "") &
    (df.review_text.notna() )
].reset_index(drop=True)


### Extract all reviews into a list

In [6]:
reviews = df_paris.review_text.tolist()

### Load an embedding model

In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")

### move model to GPU (This is common in ML)

In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

### Encode user search query & reviews

In [9]:
query_embedding = model.encode([user_query])
review_embeddings = model.encode(reviews)

### Make sure Embeddings are of type float32

In [10]:
query_embedding = query_embedding.astype(np.float32)
review_embeddings = review_embeddings.astype(np.float32)

### Normalize Embeddings

In [11]:
faiss.normalize_L2(query_embedding)
faiss.normalize_L2(review_embeddings)

### Check if dimensions of embeddings match

In [12]:
assert query_embedding.shape[1] == review_embeddings.shape[1]
embedding_dimension = query_embedding.shape[1]

## Perform Similarity Searching using FAISS

### Create an Index

In [13]:
index = faiss.IndexFlatIP(embedding_dimension)

### Add reviews

In [14]:
index.add(review_embeddings)

### Perform Similarity search

In [15]:
similarity_scores, indices = index.search(query_embedding, top_k)

### Output top-k most relevant results

In [16]:
for i, (score, idx) in enumerate(zip(similarity_scores[0], indices[0]), start=1):
    row = df_paris.iloc[idx]
    
    print(f"Query: {user_query}")
    print(f"{i}. Hotel Name: {row.hotel_name} ")
    print(f"Review: {row.review_text}")
    print(f"Distance: {score}")
    print()

Query: An affordable hotel with a view of the Eiffel Tower
1. Hotel Name: Pullman Paris Eiffel Tower Hotel 
Review: Excellent service. Stunning view of the Eiffel towere from our balcony, The room is gorgeous, comfortable and spacious. Definitely will be recommending this hotel to family and friends. If you’re looking for a hotel that has everything you need in Paris, luxury and view, this is the one.
Distance: 0.8450589776039124

Query: An affordable hotel with a view of the Eiffel Tower
2. Hotel Name: Hotel Tourisme Avenue 
Review: Nice Hotel only a few minutes away from Eiffel tower. 3 Metro lines in Front of the hotel, supermarket and restaurants very close. Small but nice and clean rooms, good bed, very friendly and helpful personal, very good breakfast buffet. 
Distance: 0.8045756816864014

Query: An affordable hotel with a view of the Eiffel Tower
3. Hotel Name: Cler Hotel 
Review: Wonderful hotel. Very close to the Eiffel Tower. Lots of restaurants and shops. Very clean. Helpfu

### increase the number of reviews to 25

In [17]:
top_k =25
distances, indices = index.search(query_embedding, top_k)

### create a dictionary to store hotel information

In [18]:
hotel_info = {}

In [31]:
for idx, distance in zip(indices[0], distances[0]):
    row = df_paris.iloc[idx]
    hotel_name = row["hotel_name"]
    if hotel_name not in hotel_info:
        hotel_info[hotel_name] = {
            "description": row["hotel_description"],
            "reviews": [],
            "distances": []
        }
    hotel_info[hotel_name]["reviews"].append(row["review_text"])
    hotel_info[hotel_name]["distances"].append(distance)

### Calculate the overall distance score for each hotel

In [36]:
for hotel_name in hotel_info:
    hotel_info[hotel_name]["overall_distance"] = np.mean(hotel_info[hotel_name]["distances"])
    hotel_info[hotel_name]["num_reviews"] = len(hotel_info[hotel_name]["reviews"] )

### Remove hotels with less than two reviews.

In [37]:
filtered_hotels = {
    hotel_name : hotel_data
    for hotel_name, hotel_data in hotel_info.items()
    if hotel_data["num_reviews"] >= 2
}

### sort the hotels based on their overall distance score in descending order

In [38]:
sorted_hotels = sorted(filtered_hotels.items(), key=lambda x: x[1]["overall_distance"], reverse=True)

In [39]:
print(f"Query: {user_query}")
print("Top hotels with similar reviews using FAISS:")
for i, (hotel_name, hotel_data) in enumerate(sorted_hotels, start=1):
    print(f"Query: {user_query}")
    print(f"Description: {hotel_data["description"]} ")
    print(f"Overall Distance: {hotel_data["overall_distance"]}:4f ")
    print(f"Number of Reviews: {len(hotel_data["reviews"] )} ")
    print("Reviews: ")
    for review in hotel_data['reviews']:
        print(f"- {review}")
    print()

Query: An affordable hotel with a view of the Eiffel Tower
Top hotels with similar reviews using FAISS:
Query: An affordable hotel with a view of the Eiffel Tower
Description: Finding an ideal charming 4 stars hotel in Paris does not have to be difficult. Welcome to Hotel Tourisme Avenue, a nice option for travelers like you. For those interested in checking out popular landmarks while visiting Paris, Hotel Tourisme Avenue is located a short distance from Champs-Elysees (1.5 mi) and Arc de Triomphe (1.7 mi). Guest rooms offer amenities such as a flat screen TV, air conditioning, and a refrigerator, and guests can go online with free wifi offered by the hotel. Hotel Tourisme Avenue features a concierge, to help make your stay more enjoyable. The property also boasts breakfast buffet. If you’re looking for a steakhouse, consider a visit to Le Relais de l'Entrecote, Il Etait Un Square, or La Poule au Pot, which are all conveniently located a short distance from Hotel Tourisme Avenue. Duri